# Notebook 00_common: Reusable Helpers, Quantum Kernels, and Statistical Audit Tools

**Auditor**: Autonomous Senior ML Research Engineer  
**Scope**: Shared infrastructure for all diabetes benchmark notebooks (00 through 10).  
**Strict Rule**: Contains only pure notebook cells; no external .py modules.

In [1]:
import os, sys, hashlib, time
import numpy as np
import pandas as pd
from scipy import stats
import sklearn
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler
from sklearn.model_selection import StratifiedKFold, GroupKFold, train_test_split
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score,
    recall_score, f1_score, roc_auc_score, average_precision_score,
    confusion_matrix, brier_score_loss
)
import pennylane as qml
import torch

GLOBAL_SEED = 42
np.random.seed(GLOBAL_SEED)
torch.manual_seed(GLOBAL_SEED)

print('Environment initialized successfully.')
print(f'NumPy: {np.__version__}, Pandas: {pd.__version__}, Scikit-learn: {sklearn.__version__}, PennyLane: {qml.__version__}, PyTorch: {torch.__version__}')

Environment initialized successfully.
NumPy: 2.2.6, Pandas: 3.0.6, Scikit-learn: 1.9.1, PennyLane: 0.44.1, PyTorch: 2.14.1+cpu


In [2]:
def load_pima(impute_zeros=False, return_raw=False):
    """
    Loads Pima Indians Diabetes Dataset (768 rows, 8 features).
    Optionally replaces invalid clinical zeros (Glucose, BP, Skin, Insulin, BMI) with column medians.
    """
    path = os.path.join('data', 'pima', 'pima-indians-diabetes.csv')
    cols = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome']
    df = pd.read_csv(path, header=None, names=cols)
    if return_raw:
        return df
    
    X = df.drop(columns=['Outcome']).copy()
    y = df['Outcome'].values.astype(int)
    
    if impute_zeros:
        zero_cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
        for col in zero_cols:
            median_val = X.loc[X[col] > 0, col].median()
            X[col] = X[col].replace(0, median_val)
            
    return X, y

def load_sylhet(return_raw=False):
    """
    Loads Early Stage Diabetes Risk Prediction (Sylhet UCI 529, 520 rows, 16 features).
    Provides exact duplicate cluster groups for GroupKFold validation.
    """
    path = os.path.join('data', 'sylhet', 'early_stage_diabetes_uci529.csv')
    df = pd.read_csv(path)
    if return_raw:
        return df
        
    y_col = [c for c in df.columns if c.lower() in ('class', 'outcome', 'target')][0]
    y = (df[y_col].astype(str).str.lower().isin(['positive', '1'])).values.astype(int)
    X_raw = df.drop(columns=[y_col]).copy()
    
    # Binary encode categorical features
    X = pd.DataFrame(index=df.index)
    for col in X_raw.columns:
        if col.lower() == 'age':
            X[col] = X_raw[col].astype(float)
        elif col.lower() == 'gender':
            X[col] = (X_raw[col].astype(str).str.lower() == 'male').astype(float)
        else:
            X[col] = (X_raw[col].astype(str).str.lower() == 'yes').astype(float)
            
    # Assign duplicate group IDs so identical rows always stay in the same fold
    _, group_ids = np.unique(X.values, axis=0, return_inverse=True)
    return X, y, group_ids

def load_cdc(n_samples=2000, seed=42, return_raw=False):
    """
    Loads CDC Diabetes Health Indicators (BRFSS UCI 891, 253,680 rows).
    Optionally returns stratified subsample of size n_samples for quantum kernels.
    """
    path = os.path.join('data', 'cdc', 'cdc_diabetes_health_indicators_uci891.csv')
    df = pd.read_csv(path)
    if return_raw and n_samples is None:
        return df
        
    y_col = 'Diabetes_binary'
    y = df[y_col].values.astype(int)
    X = df.drop(columns=[y_col, 'ID'], errors='ignore').copy()
    
    if n_samples is not None and n_samples < len(df):
        from sklearn.model_selection import StratifiedShuffleSplit
        sss = StratifiedShuffleSplit(n_splits=1, train_size=n_samples, random_state=seed)
        idx, _ = next(sss.split(X, y))
        X = X.iloc[idx].reset_index(drop=True)
        y = y[idx]
        
    return X, y

def load_130_hospitals(n_samples=2000, seed=42, return_raw=False):
    """
    Loads UCI Diabetes 130-US Hospitals (UCI 296, 101,766 rows).
    Target: 30-day readmission ('<30' vs other).
    """
    path = os.path.join('data', 'diabetes_130_hospitals', 'diabetic_data.csv')
    df = pd.read_csv(path)
    if return_raw and n_samples is None:
        return df
        
    # Readmitted within 30 days is positive class
    y = (df['readmitted'] == '<30').values.astype(int)
    
    # Select clean clinical numeric and key demographic features
    num_cols = ['time_in_hospital', 'num_lab_procedures', 'num_procedures', 
                'num_medications', 'number_outpatient', 'number_emergency', 
                'number_inpatient', 'number_diagnoses']
    X = df[num_cols].copy().astype(float)
    
    if n_samples is not None and n_samples < len(df):
        from sklearn.model_selection import StratifiedShuffleSplit
        sss = StratifiedShuffleSplit(n_splits=1, train_size=n_samples, random_state=seed)
        idx, _ = next(sss.split(X, y))
        X = X.iloc[idx].reset_index(drop=True)
        y = y[idx]
        
    return X, y

print('Dataset loaders defined: load_pima, load_sylhet, load_cdc, load_130_hospitals')

Dataset loaders defined: load_pima, load_sylhet, load_cdc, load_130_hospitals


In [3]:
def make_angle_kernel(n_qubits):
    dev = qml.device('default.qubit', wires=n_qubits)
    @qml.qnode(dev)
    def circuit(x1, x2):
        qml.AngleEmbedding(x1, wires=range(n_qubits), rotation='Y')
        qml.adjoint(qml.AngleEmbedding)(x2, wires=range(n_qubits), rotation='Y')
        return qml.probs(wires=range(n_qubits))
    return lambda x1, x2: float(circuit(x1, x2)[0])

def make_zz_kernel(n_qubits):
    dev = qml.device('default.qubit', wires=n_qubits)
    @qml.qnode(dev)
    def circuit(x1, x2):
        # ZZ feature map with Hadamard + Rz + entangling CNOT + Rz + CNOT
        for i in range(n_qubits):
            qml.Hadamard(wires=i)
            qml.RZ(2.0 * x1[i], wires=i)
        for i in range(n_qubits):
            for j in range(i + 1, n_qubits):
                qml.CNOT(wires=[i, j])
                qml.RZ(2.0 * (np.pi - x1[i]) * (np.pi - x1[j]), wires=j)
                qml.CNOT(wires=[i, j])
        # Adjoint
        for i in reversed(range(n_qubits)):
            for j in reversed(range(i + 1, n_qubits)):
                qml.CNOT(wires=[i, j])
                qml.RZ(-2.0 * (np.pi - x2[i]) * (np.pi - x2[j]), wires=j)
                qml.CNOT(wires=[i, j])
        for i in reversed(range(n_qubits)):
            qml.RZ(-2.0 * x2[i], wires=i)
            qml.Hadamard(wires=i)
        return qml.probs(wires=range(n_qubits))
    return lambda x1, x2: float(circuit(x1, x2)[0])

def make_z_kernel(n_qubits):
    dev = qml.device('default.qubit', wires=n_qubits)
    @qml.qnode(dev)
    def circuit(x1, x2):
        for i in range(n_qubits):
            qml.Hadamard(wires=i)
            qml.RZ(2.0 * x1[i], wires=i)
        for i in reversed(range(n_qubits)):
            qml.RZ(-2.0 * x2[i], wires=i)
            qml.Hadamard(wires=i)
        return qml.probs(wires=range(n_qubits))
    return lambda x1, x2: float(circuit(x1, x2)[0])

def compute_gram_matrix(kernel_func, X1, X2=None):
    """
    Computes Gram matrix between X1 and X2 (or X1 and X1 if X2 is None).
    Optimized for symmetric computation when X2 is None.
    """
    X1 = np.asarray(X1)
    n1 = len(X1)
    if X2 is None:
        K = np.ones((n1, n1), dtype=np.float64)
        for i in range(n1):
            for j in range(i + 1, n1):
                val = float(kernel_func(X1[i], X1[j]))
                K[i, j] = val
                K[j, i] = val
        return K
    else:
        X2 = np.asarray(X2)
        n2 = len(X2)
        K = np.zeros((n1, n2), dtype=np.float64)
        for i in range(n1):
            for j in range(n2):
                K[i, j] = float(kernel_func(X1[i], X2[j]))
        return K

print('PennyLane Quantum Kernel factories defined: Angle, ZZ, Z, and Gram matrix compute.')

PennyLane Quantum Kernel factories defined: Angle, ZZ, Z, and Gram matrix compute.


In [4]:
def compute_classification_metrics(y_true, y_pred, y_prob=None):
    """
    Computes a comprehensive dictionary of binary classification metrics.
    """
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    
    acc = accuracy_score(y_true, y_pred)
    bal_acc = balanced_accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    f1 = f1_score(y_true, y_pred, zero_division=0)
    
    auc = roc_auc_score(y_true, y_prob) if (y_prob is not None and len(np.unique(y_true)) > 1) else np.nan
    pr_auc = average_precision_score(y_true, y_prob) if (y_prob is not None and len(np.unique(y_true)) > 1) else np.nan
    brier = brier_score_loss(y_true, y_prob) if y_prob is not None else np.nan
    
    return {
        'accuracy': float(acc),
        'balanced_accuracy': float(bal_acc),
        'precision': float(prec),
        'recall': float(rec),
        'specificity': float(spec),
        'f1': float(f1),
        'roc_auc': float(auc),
        'pr_auc': float(pr_auc),
        'brier': float(brier),
        'tn': int(tn),
        'fp': int(fp),
        'fn': int(fn),
        'tp': int(tp)
    }

def nadeau_bengio_ttest(diffs, n_train, n_test):
    """
    Corrected resampled t-test for repeated cross-validation (Nadeau & Bengio 2003).
    diffs: array of paired metric differences across all runs and folds (e.g., 50 differences for 10x5-fold).
    """
    diffs = np.asarray(diffs)
    k_r = len(diffs)
    mean_diff = np.mean(diffs)
    var_diff = np.var(diffs, ddof=1)
    
    correction = (1.0 / k_r) + (n_test / n_train)
    corrected_var = var_diff * correction
    
    if corrected_var <= 0:
        return 0.0, 1.0
        
    t_stat = mean_diff / np.sqrt(corrected_var)
    p_val = 2.0 * (1.0 - stats.t.cdf(np.abs(t_stat), df=k_r - 1))
    return float(t_stat), float(p_val)

def holm_bonferroni(p_values):
    """
    Holm-Bonferroni correction for multiple hypothesis testing.
    """
    p_values = np.asarray(p_values)
    n = len(p_values)
    order = np.argsort(p_values)
    sorted_p = p_values[order]
    
    adjusted = np.empty(n)
    running_max = 0.0
    for i, p in enumerate(sorted_p):
        adj = p * (n - i)
        running_max = max(running_max, adj)
        adjusted[i] = min(running_max, 1.0)
        
    out = np.empty(n)
    out[order] = adjusted
    return out

def bootstrap_metric_ci(y_true, y_pred, metric_func=accuracy_score, n_boot=1000, alpha=0.05, seed=42):
    """
    Computes empirical bootstrap confidence intervals.
    """
    rng = np.random.RandomState(seed)
    n = len(y_true)
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    boot_stats = []
    
    for _ in range(n_boot):
        idx = rng.randint(0, n, size=n)
        if len(np.unique(y_true[idx])) < 2:
            continue
        val = metric_func(y_true[idx], y_pred[idx])
        boot_stats.append(val)
        
    boot_stats = np.array(boot_stats)
    low = np.percentile(boot_stats, 100.0 * (alpha / 2.0))
    high = np.percentile(boot_stats, 100.0 * (1.0 - alpha / 2.0))
    point = metric_func(y_true, y_pred)
    return float(point), float(low), float(high)

print('Statistical testing tools defined: Nadeau-Bengio corrected t-test, Holm-Bonferroni, Bootstrap CI.')

Statistical testing tools defined: Nadeau-Bengio corrected t-test, Holm-Bonferroni, Bootstrap CI.
